# Fase 1 — Planejamento do objetivo e do modelo

**Projeto:** Fine-Tuning para Triagem de Alertas em SOC

Nesta fase você vai:

1. Definir a tarefa do modelo e as metas de sucesso.
2. Escolher o modelo base.
3. Rodar o **teste de necessidade**: medir se um prompt few-shot no modelo base já resolve a triagem. Se resolver, não há por que fazer fine-tuning.

Consulte o enunciado em `reports/plano-fine-tuning-triagem-soc.md` (Fase 1), principalmente a **política de severidade** (seção 1.2) e as **metas** (seção 1.3).

**Antes de começar:**

- *Arquivo → Salvar uma cópia no Drive*. Sem isso, suas alterações se perdem.
- *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*.
- Ícone de chave (*Secrets*): crie `HF_TOKEN` com seu token de leitura do Hugging Face e libere o acesso deste notebook.

Células marcadas com `# TODO` são para você completar. Células de texto com `_____` são para você responder.

## Passo 1 — Preparar o ambiente

Instala as bibliotecas, autentica no Hugging Face e confirma que a GPU está ativa.

> Nunca cole o token direto numa célula. Se o notebook for salvo no GitHub, o token vaza.

In [ ]:
!pip install -q transformers datasets peft trl evaluate scikit-learn

from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

In [ ]:
import torch

print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

## Passo 2 — Definir a tarefa

Responda: o que o modelo recebe, o que devolve, qual o tipo de fine-tuning e o que fica **fora** do papel do modelo.

| Item | Sua definição |
|---|---|
| Entrada | _____ |
| Saída | _____ |
| Tipo de fine-tuning | _____ |
| Fora do escopo | _____ |

**Pergunta:** veredito e severidade poderiam vir de um classificador simples. Por que usar um LLM? _____

## Passo 3 — Aplicar a política de severidade à mão

Use a tabela da política (plano, seção 1.2) para triar os 4 alertas:

- **A)** EDR: `powershell.exe -enc ...` com processo pai `winword.exe` no host `DC-01` (controlador de domínio). Ocorrências em 7 dias: 1.
- **B)** EDR: execução de `psexec.exe` pelo usuário `m.souza` (analista de RH, sem privilégio admin) na estação `WS-RH-14`. Ocorrências em 7 dias: 1.
- **C)** IDS: varredura de portas a partir do scanner interno `LAB-SCAN-01` no ativo `LAB-WEB-03`. Ocorrências em 7 dias: 120, nenhum incidente associado.
- **D)** EDR: hash com reputação **maliciosa** na Threat Intel executado na máquina `LAB-TEST-07` (laboratório). Ocorrências em 7 dias: 1.

| Alerta | Sinal técnico | Ativo | Veredito | Severidade | Justificativa |
|---|---|---|---|---|---|
| A | _____ | _____ | _____ | _____ | _____ |
| B | _____ | _____ | _____ | _____ | _____ |
| C | _____ | _____ | _____ | _____ | _____ |
| D | _____ | _____ | _____ | _____ | _____ |

**Pergunta:** o que os alertas A e D têm em comum e por que a severidade deles é diferente? _____

## Passo 4 — Definir os critérios de sucesso

As metas são fixadas **antes** de treinar. Justifique cada uma.

| Métrica | Meta | Por que essa métrica |
|---|---|---|
| JSON válido com as 3 chaves | ≥ 98% | _____ |
| Acurácia de veredito | ≥ 85% | _____ |
| Recall de severidade "crítica" | ≥ 90% | _____ |
| Justificativa coerente (10 casos) | ≥ 8 de 10 | _____ |
| Ganho sobre o baseline | Superar o baseline | _____ |

**Pergunta:** por que, para a classe "crítica", o recall importa mais que a precisão? _____

## Passo 5 — Escolher o modelo base

A célula abaixo consulta no Hub a licença, o acesso *gated* e o tamanho de cada candidato. Rode e preencha a tabela.

In [ ]:
from huggingface_hub import model_info

candidatos = [
    "Qwen/Qwen2.5-0.5B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "meta-llama/Llama-3.2-1B-Instruct",
    "microsoft/Phi-3.5-mini-instruct",
]

for model_id in candidatos:
    info = model_info(model_id)
    licenca = info.card_data.license if info.card_data else None
    params = info.safetensors.total if info.safetensors else None
    print(f"{model_id:40} licença={licenca} gated={info.gated} params={params}")

| Critério | Qwen2.5-0.5B-Instruct | Qwen2.5-1.5B-Instruct | Llama-3.2-1B-Instruct | Phi-3.5-mini-instruct |
|---|---|---|---|---|
| Instruction-tuned | _____ | _____ | _____ | _____ |
| Licença | _____ | _____ | _____ | _____ |
| Acesso *gated* | _____ | _____ | _____ | _____ |
| Treina numa T4 no tempo do projeto | _____ | _____ | _____ | _____ |
| Implantação on-premises em volume | _____ | _____ | _____ | _____ |

**Modelo escolhido e por quê:** _____

**Por que os outros foram descartados:** _____

## Passo 6 — Carregar o modelo e conferir o chat template

Observe o formato que o modelo espera (tokens especiais de system/user/assistant) e quanta memória da GPU ele ocupa.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")

exemplo = [
    {"role": "system", "content": "Você é um analista de SOC."},
    {"role": "user", "content": "Alerta de teste"},
]
print(tokenizer.apply_chat_template(exemplo, tokenize=False, add_generation_prompt=True))
print(f"Memória da GPU: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

**Pergunta:** o que `add_generation_prompt=True` acrescenta no final e por que isso é necessário? _____

## Passo 7 — Montar o prompt few-shot do baseline

O baseline precisa ser **o melhor prompt razoável**, não um prompt fraco feito para perder.

Complete:

1. `POLITICA`: a política de severidade em texto e a instrução de responder **apenas** com o JSON das 3 chaves.
2. `FEW_SHOT`: **3 exemplos escritos por você**, cada um com `(texto do alerta + contexto, triagem em dicionário)`. Cubra severidades e vereditos diferentes.
3. `montar_mensagens`: devolve a lista de mensagens com o system, os 3 exemplos (user/assistant) e o alerta a triar.

> Não use alertas da amostra de baseline como exemplo. Isso vazaria a resposta e inflaria o resultado.

In [ ]:
import json

# TODO: escreva a política de severidade e a instrução de formato de saída
POLITICA = """
_____
"""

SYSTEM_BASELINE = f"Você é um analista de SOC.\n{POLITICA}"

# TODO: 3 exemplos no formato (texto_do_alerta_com_contexto, {"veredito": ..., "severidade": ..., "justificativa": ...})
FEW_SHOT = [
]


def montar_mensagens(alerta: str, contexto: str) -> list[dict]:
    # TODO: system + 3 pares user/assistant do FEW_SHOT + o alerta a triar
    # Dica: a resposta do assistant deve ser json.dumps(triagem, ensure_ascii=False)
    ...

## Passo 8 — Rodar o baseline na amostra

Carrega os 20 alertas da amostra e gera a triagem de cada um. `do_sample=False` deixa o resultado reprodutível.

In [ ]:
from datasets import load_dataset

DATASET_ID = "org-da-turma/alertas-soc"

amostra = load_dataset(DATASET_ID, data_files="amostra_baseline.jsonl", split="train")
print(len(amostra))
print(amostra[0])


def gerar(messages: list[dict]) -> str:
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
    out = model.generate(inputs, max_new_tokens=200, do_sample=False)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)


saidas = [gerar(montar_mensagens(ex["alerta"], ex["contexto"])) for ex in amostra]
print(saidas[0])

## Passo 9 — Medir o resultado do baseline

A avaliação é **estrita** e será a mesma da Fase 6: uma saída inválida conta como erro em todas as métricas.

Complete:

1. `validar`: devolve o dicionário se a saída for JSON com **exatamente** as 3 chaves e valores permitidos. Caso contrário, devolve `None`.
2. O cálculo das métricas: JSON válido, acurácia de veredito, acurácia de severidade e recall de "crítica".

In [ ]:
VEREDITOS = {"falso_positivo", "verdadeiro_positivo"}
SEVERIDADES = {"baixa", "media", "alta", "critica"}
CHAVES = {"veredito", "severidade", "justificativa"}


def validar(saida: str) -> dict | None:
    # TODO: json.loads com tratamento de erro + checagem de chaves e de valores permitidos
    ...


previstos = [validar(s) for s in saidas]
reais = [ex["triagem"] for ex in amostra]
n = len(reais)

# TODO: calcule as métricas (saída inválida conta como erro)
json_valido = ...
acc_veredito = ...
acc_severidade = ...
recall_critica = ...  # entre os alertas realmente críticos, quantos foram previstos como "critica"

print(f"JSON válido:         {json_valido:.0%}")
print(f"Acurácia veredito:   {acc_veredito:.0%}")
print(f"Acurácia severidade: {acc_severidade:.0%}")
print(f"Recall de crítica:   {recall_critica:.0%}")

In [ ]:
for s, p in zip(saidas, previstos):
    if p is None:
        print("INVÁLIDA:", s[:150])

## Passo 10 — Interpretar o resultado e decidir

| Métrica | Meta | Baseline (few-shot) | Atingiu? |
|---|---|---|---|
| JSON válido | ≥ 98% | _____ | _____ |
| Acurácia de veredito | ≥ 85% | _____ | _____ |
| Recall de "crítica" | ≥ 90% | _____ | _____ |

**Principais falhas observadas** (formato? severidade? quais tipos de alerta?): _____

**Decisão:** o fine-tuning se justifica? Por quê? _____

> Se o baseline atingir todas as metas, a conclusão correta é "não faria fine-tuning neste caso". Se estiver bem justificada, essa conclusão é válida.

## Passo 11 — Ficha da Fase 1 (entregável)

Copie e preencha:

```
# Ficha da Fase 1 — <seu nome>

## Tarefa
Entrada: _____
Saída: _____
Tipo: _____
Fora do escopo: _____

## Metas (definidas antes do treino)
- _____

## Modelo base
Modelo: _____
Licença (link): _____
Motivo: _____
Memória na T4: _____

## Baseline (few-shot, 20 alertas)
JSON válido: __% | Acurácia de veredito: __% | Recall de crítica: __%
Principais falhas observadas: _____

## Decisão
O fine-tuning [se justifica / não se justifica] porque _____.
```